# Evaluate method_a predictor accuracy

Loads an `eval__baseline__qh_predictions__{MODEL_DATA}.json` file (produced by
`create_verification_data`, run once per (TASK, MODEL_DATA)), then runs each
trained `method_a_predictors` model (one per MODEL_TRAIN) on those same
prompts, extracts the predicted binary label from `<answer>X</answer>`, and
compares against the ground-truth `label` field to compute accuracy.

Only one vllm model is ever loaded at a time (freed before the next model
loads). Eval records for a given (TASK, MODEL_DATA) are loaded once and
reused across every MODEL_TRAIN.

Run this notebook from the repo root (`abstention-reasoning/`).

In [24]:
import os

# Restrict to a single GPU before vllm/torch are imported anywhere in this
# process. Set to the device you want to use, or comment out to use whatever
# CUDA_VISIBLE_DEVICES is already set in the shell that launched Jupyter.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import re
import gc
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pipeline").exists():
    # Allow running the notebook from a subdirectory too.
    REPO_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else REPO_ROOT
sys.path.insert(0, str(REPO_ROOT))

from pipeline.core.io import load_json
from pipeline.core.generator import Generator, GenerationConfig

In [25]:
# === Config ===

# Sweep over every (task, model_data, model_train) combination. Adjust freely,
# or shrink these lists to evaluate a single combo.
TASKS = ["countdown", "competition_math"]
MODEL_DATA_LIST = ["qwen2.5-3b", "qwen3-4b-base"]
MODEL_TRAIN_LIST = ["qwen2.5-3b", "qwen2.5-1.5b"]

EVAL_FILE_TEMPLATE = "artifacts/{task}/sft_datasets_verification/eval__baseline__qh_predictions__{model_data}.json"
MODEL_PATH_TEMPLATE = "artifacts/{task}/models/method_a_predictors/{model_data}/{model_train}/model"

# Greedy decoding: this is a binary classification task, not open-ended
# generation, so temperature=0 gives the model's actual argmax prediction
# rather than a sampled one.
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NEW_TOKENS = 16  # only need "<answer>0</answer>" or "<answer>1</answer>"
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = 0.9

ANSWER_RE = re.compile(r"<answer>\s*([01])\s*</answer>")

In [26]:
def extract_predicted_label(text: str) -> int | None:
    """Parse the predicted 0/1 label out of a generated completion.

    Returns None if the model didn't emit a well-formed <answer> tag (counted
    as a wrong prediction, not silently dropped, when computing accuracy).
    """
    match = ANSWER_RE.search(text)
    if match is None:
        return None
    return int(match.group(1))


def run_predictor(model_path: str, records: list[dict]) -> list[int | None]:
    """Load one predictor model, run it over eval records, then free it.

    Only one vllm engine is ever alive at a time: the previous model (if any)
    must already be freed by the caller before this is invoked.
    """
    config = GenerationConfig(
        model_name=model_path,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS,
        num_samples=1,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        seed=42,
    )
    generator = Generator(config)
    prompts = [r["prompt"] for r in records]
    outputs = generator.generate(prompts)  # list[list[{"text": ...}]], one sample each

    predictions = [extract_predicted_label(samples[0]["text"]) for samples in outputs]

    # Explicitly release the vllm engine before the next model loads --
    # otherwise GPU memory from consecutive models in the sweep accumulates.
    del generator
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except ImportError:
        pass

    return predictions


def compute_metrics(records: list[dict], predictions: list[int | None]) -> dict:
    """Accuracy/precision/recall/F1 + parse-failure rate for one eval run."""
    labels = [r["label"] for r in records]

    tp = fp = tn = fn = parse_failures = 0
    for label, pred in zip(labels, predictions):
        if pred is None:
            parse_failures += 1
            continue  # counted as wrong below via correct-count math
        if pred == 1 and label == 1:
            tp += 1
        elif pred == 1 and label == 0:
            fp += 1
        elif pred == 0 and label == 0:
            tn += 1
        elif pred == 0 and label == 1:
            fn += 1

    n = len(records)
    correct = tp + tn
    accuracy = correct / n if n else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else float("nan")
    )

    return {
        "n": n,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "parse_failures": parse_failures,
    }

In [27]:
# === Sweep ===
# Eval records are loaded once per (task, model_data) and reused across every
# model_train. Exactly one vllm model is loaded at a time.

results = []
eval_records_cache: dict[tuple[str, str], list[dict]] = {}
# Keyed by (task, model_data, model_train) -> {"records": ..., "predictions": ...}.
# Kept around so a later cell can break accuracy down by difficulty without
# re-running generation.
detailed_results: dict[tuple[str, str, str], dict] = {}

for task in TASKS:
    for model_data in MODEL_DATA_LIST:
        eval_path = REPO_ROOT / EVAL_FILE_TEMPLATE.format(task=task, model_data=model_data)
        if not eval_path.exists():
            print(f"SKIP (missing eval file): {eval_path}")
            continue
        cache_key = (task, model_data)
        if cache_key not in eval_records_cache:
            eval_records_cache[cache_key] = load_json(eval_path)
        records = eval_records_cache[cache_key]

        for model_train in MODEL_TRAIN_LIST:
            model_path = REPO_ROOT / MODEL_PATH_TEMPLATE.format(
                task=task, model_data=model_data, model_train=model_train
            )
            if not model_path.exists():
                print(f"SKIP (missing model): {model_path}")
                continue

            print(f"=== task={task} model_data={model_data} model_train={model_train} ===")
            predictions = run_predictor(str(model_path), records)
            metrics = compute_metrics(records, predictions)
            metrics.update(task=task, model_data=model_data, model_train=model_train)
            results.append(metrics)
            detailed_results[(task, model_data, model_train)] = {
                "records": records,
                "predictions": predictions,
            }
            print(metrics)

summary = pd.DataFrame(results)
cols = ["task", "model_data", "model_train", "n", "accuracy", "precision", "recall", "f1", "parse_failures"]
summary = summary[cols] if not summary.empty else summary
summary

=== task=countdown model_data=qwen2.5-3b model_train=qwen2.5-3b ===
Loading model: /home/tanyagoyal/abstention-reasoning/artifacts/countdown/models/method_a_predictors/qwen2.5-3b/qwen2.5-3b/model
INFO 09-30 10:53:18 [utils.py:261] non-default args: {'disable_log_stats': True, 'model': '/home/tanyagoyal/abstention-reasoning/artifacts/countdown/models/method_a_predictors/qwen2.5-3b/qwen2.5-3b/model'}
INFO 09-30 10:53:18 [model.py:541] Resolved architecture: Qwen2ForCausalLM
INFO 09-30 10:53:18 [model.py:1882] Downcasting torch.float32 to torch.bfloat16.
INFO 09-30 10:53:18 [model.py:1561] Using max model len 32768
INFO 09-30 10:53:18 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:18 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='/home/tanyagoyal/abstention-reasoning/artifacts/countdown/models/method_a_predictors/qwen2.5-3b/qwen2.5-3b/model', speculative_config=None, tokenizer='/home/ta

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.52it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.22it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:02<00:00,  1.42it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:02<00:00,  1.39it/s]
(EngineCore_DP0 pid=1009195) 


(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:22 [default_loader.py:291] Loading weights took 2.29 seconds
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:23 [gpu_model_runner.py:4118] Model loading took 5.79 GiB memory and 3.042315 seconds
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:29 [backends.py:805] Using cache directory: /home/tanyagoyal/.cache/vllm/torch_compile_cache/3c242bd28a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:29 [backends.py:865] Dynamo bytecode transform time: 5.78 s
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:34 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.437 s
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:34 [monitor.py:34] torch.compile takes 7.22 s in total
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:35 [gpu_worker.py:356] Available KV cache memory: 32.73 GiB
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:35 [kv_cache_utils.py:1307] GPU KV cache size: 95

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.64it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 24.70it/s]


(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:40 [gpu_model_runner.py:5051] Graph capturing finished in 5 secs, took 0.53 GiB
(EngineCore_DP0 pid=1009195) INFO 09-30 10:53:40 [core.py:272] init engine (profile, create kv cache, warmup model) took 16.58 seconds


...

(EngineCore_DP0 pid=1012409) The tokenizer you are loading from '/home/tanyagoyal/abstention-reasoning/artifacts/competition_math/models/method_a_predictors/qwen3-4b-base/qwen2.5-1.5b/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 09-30 10:57:48 [llm.py:343] Supported tasks: ['generate']


Processed prompts: 100%|██████████| 1764/1764 [00:09<00:00, 189.47it/s, est. speed input: 44654.82 toks/s, output: 1326.42 toks/s]


{'n': 1764, 'accuracy': 0.8010204081632653, 'precision': 0.834107498341075, 'recall': 0.9256259204712813, 'f1': 0.8774869109947645, 'tp': 1257, 'fp': 250, 'tn': 156, 'fn': 101, 'parse_failures': 0, 'task': 'competition_math', 'model_data': 'qwen3-4b-base', 'model_train': 'qwen2.5-1.5b'}


,task,model_data,model_train,n,accuracy,precision,recall,f1,parse_failures
0,countdown,qwen2.5-3b,qwen2.5-3b,1500,0.758000,0.758000,1.000000,0.862344,0
1,countdown,qwen2.5-3b,qwen2.5-1.5b,1500,0.760667,0.760375,0.999120,0.863550,0
2,countdown,qwen3-4b-base,qwen2.5-3b,1500,0.811333,0.840372,0.951378,0.892436,0
3,countdown,qwen3-4b-base,qwen2.5-1.5b,1500,0.818667,0.841135,0.961102,0.897126,0
4,competition_math,qwen2.5-3b,qwen2.5-3b,1764,0.717120,0.763787,0.774464,0.769088,0
5,competition_math,qwen2.5-3b,qwen2.5-1.5b,1764,0.740930,0.752874,0.854613,0.800524,0
6,competition_math,qwen3-4b-base,qwen2.5-3b,1764,0.811224,0.832576,0.944772,0.885133,0
7,competition_math,qwen3-4b-base,qwen2.5-1.5b,1764,0.801020,0.834107,0.925626,0.877487,0


## Optional: save summary to disk

In [28]:
OUTPUT_CSV = REPO_ROOT / "artifacts" / "method_a_predictor_eval_summary.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_CSV, index=False)
print(f"Saved summary to {OUTPUT_CSV}")

Saved summary to /home/tanyagoyal/abstention-reasoning/artifacts/method_a_predictor_eval_summary.csv


## Same-model vs. different-model accuracy

Reshapes `summary` (already computed above -- this cell doesn't re-run the
sweep) into one table per task: rows are `MODEL_DATA` (the model whose
samples the predictor was trained/evaluated on), columns are every
`MODEL_TRAIN` (the predictor's backbone). The diagonal -- where
`MODEL_TRAIN == MODEL_DATA` -- is the "same model" (in-distribution) case and
is highlighted; every other cell is "different model" (cross-model).

In [31]:
def style_same_vs_different(df: pd.DataFrame, task: str):
    """Pivot accuracy to a model_data x model_train grid for one task and
    highlight the same-model diagonal. Axis names turn into header labels:
    MODEL_TRAIN sits above the column headers, MODEL_DATA sits beside the
    row index."""
    sub = df[df["task"] == task]
    pivot = sub.pivot_table(index="model_data", columns="model_train", values="accuracy")
    pivot.index.name = "MODEL_DATA"
    pivot.columns.name = "MODEL_TRAIN"

    def highlight_same_model(row):
        return [
            "background-color: #c6f6d5; font-weight: bold" if col == row.name else ""
            for col in pivot.columns
        ]

    styled = (
        pivot.style
        .apply(highlight_same_model, axis=1)
        .format("{:.1%}")
        .set_caption(f"{task} (green = same model)")
    )
    return styled


for _task in summary["task"].unique():
    display(style_same_vs_different(summary, _task))

MODEL_TRAIN,qwen2.5-1.5b,qwen2.5-3b
MODEL_DATA,,
qwen2.5-3b,76.1%,75.8%
qwen3-4b-base,81.9%,81.1%


MODEL_TRAIN,qwen2.5-1.5b,qwen2.5-3b
MODEL_DATA,,
qwen2.5-3b,74.1%,71.7%
qwen3-4b-base,80.1%,81.1%


## Accuracy by difficulty

Reads difficulty straight from the source files already on disk (no
re-generation): `level` for `competition_math` (joined in from the
`eval__baseline__verification-10s__{model_data}.internal.json` file used to
build the eval set, matched on `index`), and `variant` for `countdown`
(already present on each eval record, e.g. `5_operands`).

This uses `detailed_results` populated by the sweep cell above -- if that
dict is empty (e.g. the sweep cell hasn't been re-run since this cell was
added), re-run the sweep cell once to populate it; no model generation needs
to be re-run beyond that.

In [32]:
_level_cache: dict[tuple[str, str], dict[int, str]] = {}


def _competition_math_levels(model_data: str) -> dict[int, str]:
    """index -> ground_truth['level'], read from the internal generations
    file (not the qh_predictions file, which doesn't carry level through)."""
    key = ("competition_math", model_data)
    if key not in _level_cache:
        internal_path = (
            REPO_ROOT
            / "artifacts/competition_math/sft_datasets_verification"
            / f"eval__baseline__verification-10s__{model_data}.internal.json"
        )
        internal_records = load_json(internal_path)
        _level_cache[key] = {
            r["index"]: r["ground_truth"]["level"] for r in internal_records
        }
    return _level_cache[key]


def get_difficulty(record: dict, task: str, model_data: str) -> str:
    """Difficulty label for one eval record."""
    if task == "competition_math":
        return _competition_math_levels(model_data).get(record["index"], "unknown")
    return record.get("variant", "unknown")  # countdown: operand count, e.g. "5_operands"


def accuracy_by_difficulty(task: str, model_data: str, model_train: str) -> pd.DataFrame:
    entry = detailed_results[(task, model_data, model_train)]
    records, predictions = entry["records"], entry["predictions"]

    rows = []
    for record, pred in zip(records, predictions):
        rows.append({
            "difficulty": get_difficulty(record, task, model_data),
            "correct": pred is not None and pred == record["label"],
        })
    df = pd.DataFrame(rows)
    return (
        df.groupby("difficulty")["correct"]
        .agg(accuracy="mean", n="count")
        .reset_index()
        .sort_values("difficulty")
    )


for (task, model_data, model_train) in detailed_results:
    print(f"=== task={task} model_data={model_data} model_train={model_train} ===")
    display(accuracy_by_difficulty(task, model_data, model_train))

=== task=countdown model_data=qwen2.5-3b model_train=qwen2.5-3b ===


,difficulty,accuracy,n
0,4_operands,0.968442,507
1,5_operands,0.778234,487
2,6_operands,0.527668,506


=== task=countdown model_data=qwen2.5-3b model_train=qwen2.5-1.5b ===


,difficulty,accuracy,n
0,4_operands,0.968442,507
1,5_operands,0.778234,487
2,6_operands,0.535573,506


=== task=countdown model_data=qwen3-4b-base model_train=qwen2.5-3b ===


...

,difficulty,accuracy,n
0,Level 1,0.910000,100
1,Level 2,0.835766,274
2,Level 3,0.806268,351
3,Level 4,0.700913,438
4,Level 5,0.660000,600
5,Level ?,1.000000,1


=== task=competition_math model_data=qwen3-4b-base model_train=qwen2.5-3b ===


,difficulty,accuracy,n
0,Level 1,0.950000,100
1,Level 2,0.941606,274
2,Level 3,0.900285,351
3,Level 4,0.815068,438
4,Level 5,0.673333,600
5,Level ?,1.000000,1


=== task=competition_math model_data=qwen3-4b-base model_train=qwen2.5-1.5b ===


,difficulty,accuracy,n
0,Level 1,0.970000,100
1,Level 2,0.934307,274
2,Level 3,0.903134,351
3,Level 4,0.803653,438
4,Level 5,0.650000,600
5,Level ?,1.000000,1
